# Results & Cost Analysis

Companion analysis for the **startup-book** generator (guidelines §9 research,
§11 costs). It reads the project configuration and visualises (1) the estimated
API run cost as a function of token volume, and (2) a unit-economics sensitivity
tied to the book's *Unit Economics* chapter.

Run from the repo root with the project venv:
`uv run jupyter nbconvert --to notebook --execute notebooks/results_analysis.ipynb`

In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np

from startup_book.shared.config import ConfigManager

cfg = ConfigManager()
in_rate, out_rate = cfg.cost_rates()
model = cfg.model()
print(f"model={model}  input=${in_rate}/1M  output=${out_rate}/1M")

## 1. Estimated run cost vs token volume

With an assumed 30% prompt / 70% completion split, the cost of one book run is
$$\text{cost} = \frac{0.3\,T}{10^6}\,r_{in} + \frac{0.7\,T}{10^6}\,r_{out},$$
where $T$ is the total token count and $r_{in}, r_{out}$ are the per-million
input/output prices from `config/setup.json`.

In [ ]:
totals = np.linspace(0, 60_000, 200)
cost = (0.3 * totals / 1e6) * in_rate + (0.7 * totals / 1e6) * out_rate

fig, ax = plt.subplots(figsize=(6, 3.5))
ax.plot(totals / 1000, cost, color="#1f3b73", lw=2)
ax.set_xlabel("Total tokens (thousands)")
ax.set_ylabel("Estimated cost (USD)")
ax.set_title(f"Estimated run cost — {model}")
ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig("../results/cost_vs_tokens.pdf")
approx = (0.3 * 30000 / 1e6) * in_rate + (0.7 * 30000 / 1e6) * out_rate
print(f"~30k-token book run ≈ ${approx:.4f}")

## 2. Unit-economics sensitivity (LTV vs churn)

From the book's economics chapter, the customer lifetime value is
$$\mathrm{LTV} = \frac{\overline{\mathrm{ARPA}}\cdot m_{\text{gross}}}{c_{\text{churn}}}.$$
Here we hold $\overline{\mathrm{ARPA}}=\$50$ and $m_{\text{gross}}=0.8$ and sweep
monthly churn.

In [ ]:
churn = np.linspace(0.01, 0.10, 100)
ltv = 50 * 0.8 / churn

fig, ax = plt.subplots(figsize=(6, 3.5))
ax.plot(churn * 100, ltv, color="#2e7d57", lw=2)
ax.axhline(360, color="#c0392b", ls="--", label="3x of a $120 CAC")
ax.set_xlabel("Monthly churn (%)")
ax.set_ylabel("LTV (USD)")
ax.set_title("LTV sensitivity to churn (ARPA=$50, margin=80%)")
ax.legend(frameon=False)
ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig("../results/ltv_sensitivity.pdf")
print(f"LTV at 2% churn = ${50 * 0.8 / 0.02:.0f}")

**Takeaways.** The book run is inexpensive (cents) on `gpt-4o-mini`; cost scales
linearly with tokens, so the main lever is prompt/output length. Unit economics
are dominated by churn: halving churn doubles LTV, which is why retention
(§Product–Market Fit) precedes growth spend.